In [59]:
# -------------------------------------
# --- Go to correct starting folder ---
# -------------------------------------
# (when running jupyter lab in the browser, the notebook starts with CWD = folder where it is located, which breaks imports, ...)
import os
import pathlib

while not ((cwd := pathlib.Path(os.getcwd())) / "pyproject.toml").exists():
    os.chdir(cwd.parent)  # go 1 folder up

## Approximate log2 over [0.5, 1]  & exp2 over [0, 1] with 2nd degree polynomials

### log2 over [0.5, 1]
- we look for an expression of the form `f(x) ≈ c0 + c1*x + c2*x^2`
- such that `f(0.5) = f(1) - 1`    (continuous)
- such that `f'(0.5) = 2*f'(1)`    (smooth)
- hence `c0=k1, c1=4, c2=-4/3` with `k1` a free parameter


### exp2(x) over [0, 1]
- we look for an expression of the form `g(x) ≈ d0 + d1*x + d2*x^2`
- such that `g(0) = 0.5*g(1)`      (continuous)
- such that `g'(0) = 0.5*g'(1)`    (smooth)
- hence `d0=3*k2, d1=2*k2, d2=k2` with `k2` a free parameter

### jointly optimal

We want to minimize the sum `e_log + e_exp + e_log_exp` where
- `e_log` is maximum error of `f(x)` wrt `log2` over [0.5, 1]
- `e_exp` is maximum error of `g(x)` wrt `exp2` over [0, 1]
- `e_log_exp` is maximum error of `g(f(x)+1)/2` wrt `x` over [0.5, 1]

In [60]:
import math

from numpy import linspace

In [61]:
# -------------------------------------------------------------------------
#  log2
# -------------------------------------------------------------------------
def compute_c(_k1: float) -> list[float]:
    return [_k1, 4.0, -4.0 / 3.0]


def log2_approx(x: float, c: list[float]) -> float:
    """Only efficient for x in or close to [0.5, 1]"""
    if x <= 0:
        return -1e6
    if x < 0.5:
        return log2_approx(2 * x, c) - 1
    elif x > 1:
        return log2_approx(x / 2, c) + 1
    else:
        # f(x) - but only valid in [0.5, 1]
        return c[0] + (c[1] * x) + (c[2] * x * x)


def cost_function_log2(_k1: float, _n: int) -> float:
    c = compute_c(_k1)
    xs = linspace(0.5, 1.0, _n)
    error = max(abs(log2_approx(x, c) - math.log2(x)) for x in xs)  # absolute error
    return error


# -------------------------------------------------------------------------
#  exp2
# -------------------------------------------------------------------------
def compute_d(_k2: float) -> list[float]:
    return [3.0 * _k2, 2.0 * _k2, _k2]


def exp2_approx(x: float, d: list[float]) -> float:
    """Only efficient for x in or close to [0, 1]"""
    if x < 0:
        return exp2_approx(x + 1, d) / 2
    elif x > 1:
        return exp2_approx(x - 1, d) * 2
    else:
        # g(x) - but only valid in [0, 1]
        return d[0] + (d[1] * x) + (d[2] * x * x)


def cost_function_exp2(_k2: float, _n: int) -> float:
    d = compute_d(_k2)
    xs = linspace(0.0, 1.0, _n)
    error = max(abs(exp2_approx(x, d) - (2**x)) / (2**x) for x in xs)  # relative error
    return error


# -------------------------------------------------------------------------
#  Overall cost function
# -------------------------------------------------------------------------
def _cost_function_log2_exp2(_k1: float, _k2: float, _n: int) -> float:
    # coefficients
    c = compute_c(_k1)
    d = compute_d(_k2)

    # sample points
    xs = linspace(0.5, 1.0, _n)

    # compute error
    error = max(abs(exp2_approx(log2_approx(x, c), d) - x) for x in xs)
    return error


def _cost_function_exp2_log2(_k1: float, _k2: float, _n: int) -> float:
    # coefficients
    c = compute_c(_k1)
    d = compute_d(_k2)

    # sample points
    xs = linspace(0.0, 1.0, _n)

    # compute error
    error = max(abs(log2_approx(exp2_approx(x, d), c) - x) for x in xs)
    return error


def cost_function(_k1: float, _k2: float, _n: int) -> float:
    return (
        cost_function_log2(_k1, _n)
        + cost_function_exp2(_k2, _n)
        + _cost_function_log2_exp2(_k1, _k2, _n)
        + _cost_function_exp2_log2(_k1, _k2, _n)
    )

In [62]:
optimal_k1 = -2.5
optimal_k2 = 0.5

for i in range(60):
    # Create grid
    grid_size = 0.5**i
    k1_values = linspace(optimal_k1 - grid_size, optimal_k1 + grid_size, 11)
    k2_values = linspace(optimal_k2 - grid_size, optimal_k2 + grid_size, 11)

    # Evaluate cost function at each grid point
    optimal_cost = 1e12
    optimal_k1 = 1e3
    optimal_k2 = 1e3
    for i, k1 in enumerate(k1_values):
        for j, k2 in enumerate(k2_values):
            cost = cost_function(k1, k2, 10000)
            if cost < optimal_cost:
                optimal_cost = cost
                optimal_k1 = k1
                optimal_k2 = k2

    print(optimal_cost, f"{optimal_k1:.20f}", f"{optimal_k2:.20f}")

0.3081673541183172 -2.50000000000000000000 0.30000000000000004441
0.3081673541183172 -2.50000000000000000000 0.30000000000000004441
0.17376277118576658 -2.70000000000000017764 0.35000000000000008882
0.09413102976514931 -2.65000000000000035527 0.32500000000000006661
0.06541220173982437 -2.67500000000000026645 0.33750000000000007772
0.041031751318524934 -2.66250000000000008882 0.33125000000000009992
0.037936078661268344 -2.66875000000000017764 0.33437500000000008882
0.028878115668034308 -2.66406250000000000000 0.33281250000000006661
0.028878115668034308 -2.66406250000000000000 0.33281250000000006661
0.028545431362607048 -2.66484375000000017764 0.33320312500000004441
0.027740297596161985 -2.66445312500000008882 0.33300781250000005551
0.027682096254962422 -2.66464843749999991118 0.33310546875000007772
0.027515481128970797 -2.66445312500000008882 0.33305664062500006661
0.027500586360236178 -2.66447753906250017764 0.33305664062500006661
0.027453851512809515 -2.66448974609375000000 0.33306884

In [65]:
# --- log2 ------------------------------------------------
c_opt = compute_c(float(optimal_k1))
print("log2 approx coefficients:", c_opt)
max_error_log2 = cost_function_log2(optimal_k1, 10000)
print("max abs error log2 approx over [0.5, 1]:", max_error_log2)
print()

# --- exp2 ------------------------------------------------
d_opt = compute_d(float(optimal_k2))
print("exp2 approx coefficients:", d_opt)
max_error_exp2 = cost_function_exp2(optimal_k2, 10000)
print("max rel error exp2 approx over [0, 1]:", max_error_exp2)
print()

# --- combined --------------------------------------------
max_error_log2_exp2 = _cost_function_log2_exp2(optimal_k1, optimal_k2, 10000)
max_error_exp2_log2 = _cost_function_exp2_log2(optimal_k1, optimal_k2, 10000)

print("max abs error exp2(log2(x)) approx over [0.5, 1]:", max_error_log2_exp2)
print("max abs error log2(exp2(x)) approx over [0.0, 1]:", max_error_exp2_log2)

log2 approx coefficients: [-2.664483083612307, 4.0, -1.3333333333333333]
max abs error log2 approx over [0.5, 1]: 0.00752509857416972

exp2 approx coefficients: [0.9992006862217113, 0.6661337908144742, 0.3330668954072371]
max rel error exp2 approx over [0, 1]: 0.002666246070267571

max abs error exp2(log2(x)) approx over [0.5, 1]: 0.00651277853362997
max abs error log2(exp2(x)) approx over [0.0, 1]: 0.010745871383234573
